# Module 2.1: Tokenization & Embeddings

In the previous module, we learned how Transformers do math. But text is made of letters, not numbers. We must translate human language into a format the network can compute. This process has two stages: **Tokenization** (chunking) and **Embedding** (vectorizing).

## 1. Tokenization: Text to Chunks

### The Concept
We could split text by single characters ('a', 'b', 'c'), or by whole words ('apple', 'banana'). Modern LLMs use a middle-ground called **Sub-word Tokenization** (like Byte-Pair Encoding or BPE). Common words stay whole ('apple'), but rare words get split into common chunks ('un', 'believ', 'able').

### Why?
1. **Character-level** is too dense: The word 'apple' would require 5 steps of computation just to understand one piece of fruit. The model loses context over long distances.
2. **Word-level** is too sparse: The English dictionary is essentially infinite (e.g., 'Google', 'Googling', 'Googled'). The vocabulary matrix would be too huge to fit in GPU memory.
3. **Sub-word (BPE)** is the sweet spot: It handles unknown words gracefully by breaking them down, while keeping the total vocabulary size manageable (usually around 50k to 100k tokens).

In [ ]:
import torch
import tiktoken

# Reproducibility: seed everything so random weights are identical on every run.
torch.manual_seed(0)

# GPT-4's tokenizer (cl100k_base)
encoder = tiktoken.get_encoding("cl100k_base")

text = "Transformers are unbelievably fast!"
tokens = encoder.encode(text)

print(f"Raw String: '{text}'")
print(f"Token IDs (Integers): {tokens}")

print("\nBreaking it down:")
for token_id in tokens:
    print(f"{token_id} -> '{encoder.decode([token_id])}'")

## 2. The Lookup Table: Input IDs to Vectors

### The Problem
We now have integers representing words (e.g., ID `45` and `12933`). A neural network *can* multiply these integers — that's not the issue. The real problem is that the integer itself is **meaningless** and implies a false ordering. ID `12933` is not "more" than ID `45` in any semantic sense, and `12933` is not "closer in meaning" to `12934` than it is to `45`. The IDs are just arbitrary row numbers in a dictionary. We need to translate each ID into a continuous mathematical space where distances *do* carry meaning.

### The Analogy (What is an "Embedding Dimension"?)
Imagine trying to describe a person using just an ID number (e.g., Person 45). That tells you nothing about who they are. Instead, you could describe them using 3 **dimensions** (traits): `[Friendliness, Intelligence, Athleticism]`.

An **Embedding Dimension** is exactly this. In LLMs, we choose a number (e.g., 256) and declare: *"Every word in our dictionary will be described by 256 different unknown mathematical traits."* The neural network will figure out what those 256 traits mean during training. One dimension might learn to track 'gender', another might track 'plurality' (cat vs cats), and another might track 'royalty' (King vs Man).

### The Math
We use `torch.nn.Embedding`. If our vocabulary size is $V$ (50,000 words) and our chosen embedding dimension is $D$ (256 traits), the Embedding layer is just a giant matrix of size $(V \times D)$. Giving it the ID `45` simply plucks out row number `45`, returning a vector of 256 numbers.

### Why do we need it?
By replacing a meaningless integer with a dense vector of traits, the model can place the word in a semantic "concept space". The goal is that words with similar meanings end up with similar embedding vectors. As the model trains using Backpropagation, it physically adjusts these embedding weights, gradually moving similar words closer together in this mathematical space.

> ⚠️ **Important:** A freshly created `nn.Embedding` layer is filled with **random** numbers. The semantic relationships ("cat" near "dog") only *emerge after training*. We will demonstrate this honestly below — first the random layer, then a hand-crafted "already trained" example so you can actually see the effect.

In [ ]:
import torch.nn as nn

# Setup: A vocabulary of 50,000 possible tokens. Each token gets a 256-dimension vector.
vocab_size = 50000
embedding_dim = 256

token_embedder = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

# Create a fake "batch" of data.
# Shape: (Batch_Size, Sequence_Length) -> (2 sentences, 4 words each)
input_ids = torch.tensor([
    [45, 102, 900, 3],   # Sentence 1
    [8, 12933, 4, 1]     # Sentence 2
])

# Translation step!
embedded_output = token_embedder(input_ids)

print("Dimensionality Tracking 🔥")
print(f"Input Shape: {input_ids.shape} -> (Batch, Seq_Length)")
print(f"Output Shape: {embedded_output.shape} -> (Batch, Seq_Length, Embed_Dim)")

# Let's actually LOOK at a vector. This is the dense 256-number "trait" vector for
# the very first token (ID 45). We print the first 8 traits so it fits on screen.
print(f"\nFirst 8 traits of token ID 45:\n{embedded_output[0, 0, :8].detach()}")
print("\nNote: these numbers are RANDOM right now (the layer is untrained).")

### Seeing semantics: "cat" ≈ "dog" ≫ "cat" vs "car"

The random layer above can't show us meaning yet. So let's **hand-craft** a tiny embedding table the way a *trained* model might end up — where animals point in a similar direction and vehicles point elsewhere. Then we measure **cosine similarity** (the angle between vectors: 1.0 = identical direction, 0.0 = unrelated, -1.0 = opposite).

This is what training is *trying* to achieve. We're skipping ahead to the finished result so the idea is concrete.

In [ ]:
import torch.nn.functional as F

# Hand-crafted "post-training" embeddings (4-D for readability).
# Think of the 4 dimensions loosely as: [is_animal, is_vehicle, is_furry, has_wheels]
trained_vectors = {
    "cat": torch.tensor([0.9, 0.0, 0.8, 0.0]),
    "dog": torch.tensor([0.95, 0.0, 0.7, 0.0]),
    "car": torch.tensor([0.0, 0.9, 0.0, 0.95]),
}

def cosine(a, b):
    # Cosine similarity = dot product of the unit vectors.
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

print(f"cosine(cat, dog) = {cosine(trained_vectors['cat'], trained_vectors['dog']):.3f}  <- similar meaning")
print(f"cosine(cat, car) = {cosine(trained_vectors['cat'], trained_vectors['car']):.3f}  <- unrelated meaning")
print("\nThe two animals point in nearly the same direction (~1.0),")
print("while cat vs car are almost perpendicular (~0.0). THAT is semantic structure.")

## 3. Under the Hood (One-Hot Encoding)

### The Concept
A "lookup table" sounds like standard programming, not Calculus. How does Backpropagation push gradients through a table lookup?

### The Math
Mathematically, selecting a row from a matrix is exactly equivalent to performing a **Matrix Multiplication** between the embedding matrix and a **One-Hot Encoded** vector (a vector of all zeros except for a single '1' at the target index).

### Why?
By framing the lookup as a Matrix Multiplication, the entire operation becomes differentiable. The gradients from the Error/Loss can flow seamlessly back into the Embedding Matrix, adjusting the values of the vectors so the model learns semantic relationships.

In [ ]:
import torch.nn.functional as F

# Let's use a tiny vocabulary of 5 words, and a dimension of 3 (for easy printing)
tiny_vocab = 5
tiny_dim = 3

# Our raw embedding matrix (weights)
emb_layer = nn.Embedding(tiny_vocab, tiny_dim)

# Let's say we want to embed the integer identity '2'
word_id = torch.tensor([2])

# --- METHOD 1: The Fast Lookup ---
lookup_result = emb_layer(word_id)

# --- METHOD 2: The Math Way (One-Hot + MatMul) ---
# 1. Create a One-Hot vector: [0., 0., 1., 0., 0.]
one_hot = F.one_hot(word_id, num_classes=tiny_vocab).float()

# 2. Multiply by the embedding weights matrix
matmul_result = torch.matmul(one_hot, emb_layer.weight)

print("Lookup Method Output: ", lookup_result.detach())
print("MatMul Method Output: ", matmul_result.detach())
print("\nAre they mathematically identical? ->", torch.allclose(lookup_result, matmul_result))

### 🏋️ Try it yourself

1. **Count the tokens.** Tokenize the word `"antidisestablishmentarianism"` with the `encoder` from above. How many tokens does this single long word become? Print each token's text so you can see *where* BPE chose to split it.
2. **(Bonus)** Add a `"truck"` vector to the hand-crafted `trained_vectors` dict and check that `cosine(car, truck)` is high while `cosine(cat, truck)` is low.

In [ ]:
# Task 1: Tokenize a long, rare word and count its tokens.
long_word = "antidisestablishmentarianism"
my_tokens = encoder.encode(long_word)

print(f"'{long_word}' becomes {len(my_tokens)} tokens:")
for tid in my_tokens:
    print(f"  {tid} -> '{encoder.decode([tid])}'")

# Task 2 (bonus): add your own vector and compare.
# trained_vectors["truck"] = torch.tensor([0.0, 0.95, 0.0, 0.9])
# print(cosine(trained_vectors["car"], trained_vectors["truck"]))
# print(cosine(trained_vectors["cat"], trained_vectors["truck"]))